# K-Means Clustering

Estimated time needed: **25-30** minutes

## Objectives

After completing this lab you will be able to:

*   Use scikit-learn's K-Means Clustering to cluster data


## Introduction

There are many models for **clustering** out there. In this notebook, we will be presenting the model that is considered one of the simplest models amongst them. Despite its simplicity, the **K-means** is vastly used for clustering in many data science applications, it is especially useful if you need to quickly discover insights from **unlabeled data**. In this notebook, you will learn how to use k-Means for customer segmentation.

Some real-world applications of k-means:

*   Customer segmentation
*   Understand what the visitors of a website are trying to accomplish
*   Pattern recognition
*   Machine learning
*   Data compression

In this notebook we practice k-means clustering with 2 examples:

*   k-means on a random generated dataset
*   Using k-means for customer segmentation


<h1>Table of contents</h1>

<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ul>
        <li><a href="https://#random_generated_dataset">k-Means on a randomly generated dataset</a></li>
            <ol>
                <li><a href="https://#setting_up_K_means">Setting up K-Means</a></li>
                <li><a href="https://#creating_visual_plot">Creating the Visual Plot</a></li>
            </ol>
        <p></p>
        <li><a href="https://#customer_segmentation_K_means">Customer Segmentation with K-Means</a></li>
            <ol>
                <li><a href="https://#pre_processing">Pre-processing</a></li>
                <li><a href="https://#modeling">Modeling</a></li>
                <li><a href="https://#insights">Insights</a></li>
            </ol>
    </ul>
</div>
<br>
<hr>


### Import the Libraries

Let's first import the required libraries.
Also run <b> %matplotlib inline </b> since we will be plotting in this section.


In [1]:
#you are running the lab in your  browser, so we will install the libraries using ``piplite``
# import piplite
# await piplite.install(['pandas'])
# await piplite.install(['matplotlib'])
# await piplite.install(['scipy'])
# await piplite.install(['seaborn'])


In [ ]:
import random                          # standard Python random number generator (not used directly here but useful for reproducibility)
import numpy as np                      # numerical computing: arrays, math operations
import matplotlib.pyplot as plt         # plotting library for creating charts and graphs
from sklearn.cluster import KMeans      # K-Means clustering algorithm from scikit-learn
from sklearn.datasets import make_blobs # helper to generate random clustered data for testing
%matplotlib inline                      # render plots directly inside the notebook (not in a separate window)

### Download the Data


In [ ]:
# NOTE: pyodide is only available in browser-based Jupyter (JupyterLite).
# When running locally, comment this cell out and download the CSV manually instead.
# from pyodide.http import pyfetch
#
# async def download(url, filename):
#     response = await pyfetch(url)
#     if response.status == 200:
#         with open(filename, "wb") as f:
#             f.write(await response.bytes())

<h1 id="random_generated_dataset">k-Means on a randomly generated dataset</h1>

Let's create our own dataset for this lab!


First we need to set a random seed. Use <b>numpy's random.seed()</b> function, where the seed will be set to <b>0</b>.


In [ ]:
# Fix the random seed so that results are reproducible across runs.
# Without this, make_blobs would generate different random data every time.
np.random.seed(0)

Next we will be making <i> random clusters </i> of points by using the <b> make_blobs </b> class. The <b> make_blobs </b> class can take in many inputs, but we will be using these specific ones. <br> <br> <b> <u> Input </u> </b>

<ul>
    <li> <b>n_samples</b>: The total number of points equally divided among clusters. </li>
    <ul> <li> Value will be: 5000 </li> </ul>
    <li> <b>centers</b>: The number of centers to generate, or the fixed center locations. </li>
    <ul> <li> Value will be: [[4, 4], [-2, -1], [2, -3],[1,1]] </li> </ul>
    <li> <b>cluster_std</b>: The standard deviation of the clusters. </li>
    <ul> <li> Value will be: 0.9 </li> </ul>
</ul>
<br>
<b> <u> Output </u> </b>
<ul>
    <li> <b>X</b>: Array of shape [n_samples, n_features]. (Feature Matrix)</li>
    <ul> <li> The generated samples. </li> </ul> 
    <li> <b>y</b>: Array of shape [n_samples]. (Response Vector)</li>
    <ul> <li> The integer labels for cluster membership of each sample. </li> </ul>
</ul>


In [ ]:
# Generate a synthetic dataset with 4 clearly separated clusters.
# n_samples=5000   → total number of data points spread equally across clusters
# centers          → coordinates of the 4 cluster centers in 2D space
# cluster_std=0.9  → how spread out (wide) each cluster is; smaller = tighter blobs
# X → shape (5000, 2): the 2D coordinates of each data point
# y → shape (5000,)  : the true cluster label (0-3) for each point
X, y = make_blobs(n_samples=5000, centers=[[4,4], [-2, -1], [2, -3], [1, 1]], cluster_std=0.9)
X.shape, y.shape

Display the scatter plot of the randomly generated data.


In [ ]:
# Visualize the raw data before clustering.
# X[:, 0] → x-axis values (first feature of each point)
# X[:, 1] → y-axis values (second feature of each point)
# marker='.' → small dot for each point; good for large datasets
# At this stage no colors per cluster — we just see the shape of the data.
plt.scatter(X[:, 0], X[:, 1], marker='.')

<h2 id="setting_up_K_means">Setting up K-Means</h2>
Now that we have our random data, let's set up our K-Means Clustering.


The KMeans class has many parameters that can be used, but we will be using these three:

<ul>
    <li> <b>init</b>: Initialization method of the centroids. </li>
    <ul>
        <li> Value will be: "k-means++" </li>
        <li> k-means++: Selects initial cluster centers for k-mean clustering in a smart way to speed up convergence.</li>
    </ul>
    <li> <b>n_clusters</b>: The number of clusters to form as well as the number of centroids to generate. </li>
    <ul> <li> Value will be: 4 (since we have 4 centers)</li> </ul>
    <li> <b>n_init</b>: Number of time the k-means algorithm will be run with different centroid seeds. The final results will be the best output of n_init consecutive runs in terms of inertia. </li>
    <ul> <li> Value will be: 12 </li> </ul>
</ul>

Initialize KMeans with these parameters, where the output parameter is called <b>k_means</b>.


In [ ]:
# Create the K-Means model (does NOT train yet — just configures it).
# init="k-means++"  → smart centroid initialisation: spreads starting centers apart
#                     to avoid bad random starts that slow convergence or find poor clusters
# n_clusters=4      → we expect 4 groups (matches the 4 centers used in make_blobs)
# n_init=12         → run the full algorithm 12 times with different random seeds,
#                     then keep the run with the lowest inertia (within-cluster variance)
k_means = KMeans(init = "k-means++", n_clusters = 4, n_init = 12)

Now let's fit the KMeans model with the feature matrix we created above, <b> X </b>.


In [ ]:
# Train the model: iteratively assign each point to its nearest centroid,
# then move centroids to the mean of their assigned points, until convergence.
# X is our feature matrix (5000 × 2). No labels are passed — this is unsupervised.
k_means.fit(X)

Now let's grab the labels for each point in the model using KMeans' <b> .labels\_ </b> attribute and save it as <b> k_means_labels </b>.


In [ ]:
# .labels_ is a 1D array of length n_samples.
# Each value is the cluster ID (0, 1, 2, or 3) assigned to that data point.
# Example: labels_[0] = 2 means the first data point was assigned to cluster 2.
k_means_labels = k_means.labels_
k_means_labels

We will also get the coordinates of the cluster centers using KMeans' <b> .cluster_centers\_ </b> and save it as <b> k_means_cluster_centers </b>.


In [ ]:
# .cluster_centers_ is a 2D array of shape (n_clusters, n_features).
# Each row is the (x, y) coordinate of a cluster centroid — the "average" point of that cluster.
# These centroids are what K-Means converged to after training.
k_means_cluster_centers = k_means.cluster_centers_
k_means_cluster_centers

<h2 id="creating_visual_plot">Creating the Visual Plot</h2>

So now that we have the random data generated and the KMeans model initialized, let's plot them and see what it looks like!


Please read through the code and comments to understand how to plot the model.


In [ ]:
# Initialize the plot with the specified dimensions.
fig = plt.figure(figsize=(6, 4))

# Colors uses a color map, which will produce an array of colors based on
# the number of labels there are. We use set(k_means_labels) to get the
# unique labels.
colors = plt.cm.Spectral(np.linspace(0, 1, len(set(k_means_labels))))

# Create a plot
ax = fig.add_subplot(1, 1, 1)

# For loop that plots the data points and centroids.
# k will range from 0-3, which will match the possible clusters that each
# data point is in.
for k, col in zip(range(len([[4,4], [-2, -1], [2, -3], [1, 1]])), colors):

    # Create a list of all data points, where the data points that are 
    # in the cluster (ex. cluster 0) are labeled as true, else they are
    # labeled as false.
    my_members = (k_means_labels == k)
    
    # Define the centroid, or cluster center.
    cluster_center = k_means_cluster_centers[k]
    
    # Plots the datapoints with color col.
    ax.plot(X[my_members, 0], X[my_members, 1], 'w', markerfacecolor=col, marker='.')
    
    # Plots the centroids with specified color, but with a darker outline
    ax.plot(cluster_center[0], cluster_center[1], 'o', markerfacecolor=col,  markeredgecolor='k', markersize=6)

# Title of the plot
ax.set_title('KMeans')

# Remove x-axis ticks
ax.set_xticks(())

# Remove y-axis ticks
ax.set_yticks(())

# Show the plot
plt.show()


## Practice

Try to cluster the above dataset into 3 clusters.\
Notice: do not generate the data again, use the same dataset as above.


In [ ]:
# Create a KMeans model with 3 clusters
# init="k-means++" → smart centroid initialisation (faster convergence, better results than random)
# n_clusters=3     → number of clusters to find
# n_init=12        → run the algorithm 12 times with different seeds, keep the best result
k_means3 = KMeans(init="k-means++", n_clusters=3, n_init=12)

# Train the model on X (unsupervised — no labels needed)
# After this, k_means3.labels_ contains the cluster ID (0, 1, or 2) for each data point
k_means3.fit(X)

# Create a blank figure canvas of size 6×4 inches
fig = plt.figure(figsize=(6, 4))

# Generate one distinct color per cluster using the Spectral colormap
# np.linspace(0, 1, n) → evenly spaced values between 0 and 1 (one per cluster)
# set(k_means3.labels_) → unique cluster IDs → {0, 1, 2}
colors = plt.cm.Spectral(np.linspace(0, 1, len(set(k_means3.labels_))))

# Add a single subplot (1 row, 1 col, position 1) to the figure
ax = fig.add_subplot(1, 1, 1)

# Loop over each cluster k and its assigned color col
for k, col in zip(range(len(k_means3.cluster_centers_)), colors):

    # Boolean mask: True for every data point that belongs to cluster k
    my_members = (k_means3.labels_ == k)

    # Get the centroid (center point) of cluster k — shape: (n_features,)
    cluster_center = k_means3.cluster_centers_[k]

    # Plot all data points in this cluster
    # X[my_members, 0] → x-coordinates of points in cluster k
    # X[my_members, 1] → y-coordinates of points in cluster k
    # 'w' → white line (no line drawn between points)
    # markerfacecolor=col → fill the dot with the cluster color
    # marker='.' → small dot marker
    ax.plot(X[my_members, 0], X[my_members, 1], 'w',
            markerfacecolor=col, marker='.')

    # Plot the centroid of this cluster as a larger circle
    # 'o'             → circle marker
    # markerfacecolor → fill with cluster color
    # markeredgecolor='k' → black border around the centroid marker
    # markersize=6    → larger than the data point dots
    ax.plot(cluster_center[0], cluster_center[1], 'o',
            markerfacecolor=col, markeredgecolor='k', markersize=6)

# Render and display the final plot
plt.show()

<details><summary>Click here for the solution</summary>

```python
k_means3 = KMeans(init = "k-means++", n_clusters = 3, n_init = 12)
k_means3.fit(X)
fig = plt.figure(figsize=(6, 4))
colors = plt.cm.Spectral(np.linspace(0, 1, len(set(k_means3.labels_))))
ax = fig.add_subplot(1, 1, 1)
for k, col in zip(range(len(k_means3.cluster_centers_)), colors):
    my_members = (k_means3.labels_ == k)
    cluster_center = k_means3.cluster_centers_[k]
    ax.plot(X[my_members, 0], X[my_members, 1], 'w', markerfacecolor=col, marker='.')
    ax.plot(cluster_center[0], cluster_center[1], 'o', markerfacecolor=col,  markeredgecolor='k', markersize=6)
plt.show()

```

</details>


<h1 id="customer_segmentation_K_means">Customer Segmentation with K-Means</h1>

Imagine that you have a customer dataset, and you need to apply customer segmentation on this historical data.
Customer segmentation is the practice of partitioning a customer base into groups of individuals that have similar characteristics. It is a significant strategy as a business can target these specific groups of customers and effectively allocate marketing resources. For example, one group might contain customers who are high-profit and low-risk, that is, more likely to purchase products, or subscribe for a service. A business task is to retain those customers. Another group might include customers from non-profit organizations and so on.

Let's download the datasetfrom IBM Object Storage.  **Did you know?** When it comes to Machine Learning, you will likely be working with large datasets. As a business, where can you host your data? IBM is offering a unique opportunity for businesses, with 10 Tb of IBM Cloud Object Storage: [Sign up now for free](http://cocl.us/ML0101EN-IBM-Offer-CC)


In [ ]:
# URL pointing to the customer segmentation CSV file hosted on IBM Cloud Object Storage.
# We'll use this path to load the data directly into a pandas DataFrame.
path='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%204/data/Cust_Segmentation.csv'

### Load Data From CSV File

Before you can work with the data, you must use the URL to get the Cust_Segmentation.csv.


In [ ]:
# await download(path, "Cust_Segmentation.csv")
# filename ="Cust_Segmentation.csv"


we create a pandas dataframe


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the customer dataset from the local CSV file into a pandas DataFrame.
# Each row represents one customer with demographic and financial attributes.
cust_df = pd.read_csv("Cust_Segmentation.csv")
cust_df.head()   # preview the first 5 rows

<h2 id="pre_processing">Pre-processing</h2


As you can see, **Address** in this dataset is a categorical variable. The k-means algorithm isn't directly applicable to categorical variables because the Euclidean distance function isn't really meaningful for discrete variables. So, let's drop this feature and run clustering.


In [6]:
# Drop the 'Address' column from the DataFrame
# axis=1 means we are dropping a column (axis=0 would drop a row)
# The result is stored in a new DataFrame 'df' — the original 'cust_df' is NOT modified
df = cust_df.drop('Address', axis=1)
df.head()

,Customer Id,Age,Edu,Years Employed,Income,Card Debt,Other Debt,Defaulted,DebtIncomeRatio
0,1,41,2,6,19,0.124,1.073,0.0,6.3
1,2,47,1,26,100,4.582,8.218,0.0,12.8
2,3,33,2,10,57,6.111,5.802,1.0,20.9
3,4,29,2,4,19,0.681,0.516,0.0,6.3
4,5,47,1,31,253,9.308,8.908,0.0,7.2


#### Normalizing over the standard deviation

Now let's normalize the dataset. But why do we need normalization in the first place? Normalization is a statistical method that helps mathematical-based algorithms to interpret features with different magnitudes and distributions equally. We use **StandardScaler()** to normalize our dataset.


In [8]:
from sklearn.preprocessing import StandardScaler

# Extract all rows and columns starting from index 1 (skipping column 0)
# This removes the first column (usually an ID or non-numeric column) from the feature set
# .values converts the DataFrame to a raw NumPy array
X = df.values[:, 1:]

# Replace any NaN (missing) or infinite values with 0
# This is necessary because StandardScaler cannot handle NaN values
X = np.nan_to_num(X)

# Apply Standard Scaling to the feature matrix X:
# - StandardScaler() creates a new scaler instance
# - fit_transform(X) does two things in one step:
#     1. fit:      learns the mean and std of each feature from X
#     2. transform: subtracts the mean and divides by the std for each feature
# Result: every feature has mean = 0 and standard deviation = 1
# This is critical before K-Means because it uses Euclidean distance —
# features with larger ranges would otherwise dominate the clustering
Clus_dataSet = StandardScaler().fit_transform(X)

# Display the scaled dataset
Clus_dataSet

array([[ 0.74291541,  0.31212243, -0.37878978, ..., -0.59048916,
        -0.52379654, -0.57652509],
       [ 1.48949049, -0.76634938,  2.5737211 , ...,  1.51296181,
        -0.52379654,  0.39138677],
       [-0.25251804,  0.31212243,  0.2117124 , ...,  0.80170393,
         1.90913822,  1.59755385],
       ...,
       [-1.24795149,  2.46906604, -1.26454304, ...,  0.03863257,
         1.90913822,  3.45892281],
       [-0.37694723, -0.76634938,  0.50696349, ..., -0.70147601,
        -0.52379654, -1.08281745],
       [ 2.1116364 , -0.76634938,  1.09746566, ...,  0.16463355,
        -0.52379654, -0.2340332 ]], shape=(850, 8))

<h2 id="modeling">Modeling</h2>


In our example (if we didn't have access to the k-means algorithm), it would be the same as guessing that each customer group would have certain age, income, education, etc, with multiple tests and experiments. However, using the K-means clustering we can do all this process much easier.

Let's apply k-means on our dataset, and take a look at cluster labels.


In [ ]:
# Choose the number of clusters (customer segments) to find.
clusterNum = 3

# Build and train the K-Means model on the raw (unscaled) feature matrix X.
# Note: using unscaled X here. For better results consider using Clus_dataSet (scaled).
k_means = KMeans(init = "k-means++", n_clusters = clusterNum, n_init = 12)
k_means.fit(X)

# Retrieve the cluster assignment for every customer (array of 0, 1, or 2).
labels = k_means.labels_
print(labels)

<h2 id="insights">Insights</h2>

We assign the labels to each row in the dataframe.


In [ ]:
# Add the cluster labels as a new column in the DataFrame so each customer
# is tagged with their segment ID (0, 1, or 2).
df["Clus_km"] = labels
df.head(5)

We can easily check the centroid values by averaging the features in each cluster.


In [ ]:
# Compute the mean of each feature grouped by cluster.
# This reveals the "profile" of each customer segment —
# e.g. cluster 0 might have high income and high age, cluster 2 might be young with low income.
df.groupby('Clus_km').mean()

Now, let's look at the distribution of customers based on their age and income:


In [ ]:
# Use the Education column (X[:, 1]) squared as bubble size —
# larger education level → larger circle, adding a third visual dimension.
area = np.pi * ( X[:, 1])**2

# 2D scatter plot: Age (x-axis) vs Income (y-axis).
# c=labels → color each point by its cluster ID.
# alpha=0.5 → semi-transparent points so overlapping clusters are visible.
plt.scatter(X[:, 0], X[:, 3], s=area, c=labels.astype(float), alpha=0.5)
plt.xlabel('Age', fontsize=18)
plt.ylabel('Income', fontsize=16)
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np
# Sample example if X and labels aren't defined:
# X = np.random.rand(100, 4) * 100
# labels = np.random.randint(0, 3, size=100)
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.view_init(elev=48, azim=134)
# Set axis labels
ax.set_xlabel('Education')
ax.set_ylabel('Age')
ax.set_zlabel('Income')
# 3D scatter plot
ax.scatter(X[:, 1], X[:, 0], X[:, 3], c=labels.astype(float), cmap='viridis')
plt.show()


k-means will partition your customers into mutually exclusive groups, for example, into 3 clusters. The customers in each cluster are similar to each other demographically.
Now we can create a profile for each group, considering the common characteristics of each cluster.
For example, the 3 clusters can be:

*   AFFLUENT, EDUCATED AND OLD AGED
*   MIDDLE AGED AND MIDDLE INCOME
*   YOUNG AND LOW INCOME


### Thank you for completing this lab!